# Data Share Asset Tree

This notebook creates a **Data Share** asset tree that mirrors the structure of your existing **Example** tree. For each area it:

1. Adds a **Shared Timerange** calculated condition with 48-hour capsules (example: 2 random capsules in the last 2 weeks).
2. Creates calculated signals that show only data **within** those capsules: `$signal.within($condition)`.
3. Applies an ACL so an application (user) can access the Data Share assets and signals.

**Prerequisites:** Run in Seeq Data Lab (authenticated) or configure login below. The **Example** asset tree must exist; structure (towers, areas, signals) is discovered dynamically.

## 1. Setup and configuration

In [ ]:
import os
import random
from datetime import datetime, timedelta, timezone
from typing import Optional

import pandas as pd
from seeq import spy

spy.options.compatibility = 199

# Data Share tree configuration
DATA_SHARE_ROOT = "Data Share"
WORKBOOK = "Data Share"  # Workbook for the new tree
DATASOURCE = "Data Share"  # Datasource for pushed items

# Source tree (must already exist in Seeq); structure is discovered dynamically below
EXAMPLE_ROOT = "Example"

# Shared Timerange: 48-hour capsules; example uses 2 random capsules in last 2 weeks
CAPSULE_DURATION_HOURS = 48
NUM_CAPSULES_PER_AREA = 2
LOOKBACK_DAYS = 14

# ACL: application (user) to grant access to Data Share (set to your application username)
DATA_SHARE_ACL_USER = os.environ.get("DATA_SHARE_ACL_USER", "data.share.app@company.com")

In [ ]:
# Log in to Seeq (skip if already in Seeq Data Lab)
if not spy.session.client:
    spy.login(
        url=os.environ.get("SEEQ_URL"),
        access_key=os.environ.get("SEEQ_ACCESS_KEY"),
        password=os.environ.get("SEEQ_PASSWORD"),
        # Or: credentials_file="credentials.key"
    )

### Discover source tree structure

Pull the **Example** tree from Seeq and derive towers, areas, and signal names from it. No hardcoded list of areas—everything is driven by the source tree.

In [ ]:
# Pull the source (Example) tree from Seeq and discover structure dynamically
example_tree = spy.assets.Tree(EXAMPLE_ROOT)
items_df = example_tree.items()

# Direct children of root = tower assets (e.g. Cooling Tower 1, Cooling Tower 2)
tower_assets = items_df[(items_df["Type"] == "Asset") & (items_df["Path"] == EXAMPLE_ROOT)]
tower_names = tower_assets["Name"].tolist()

# Under each tower, direct children = area assets
areas_by_tower = {}
for tower in tower_names:
    area_assets = items_df[
        (items_df["Type"] == "Asset")
        & (items_df["Path"] == f"{EXAMPLE_ROOT} >> {tower}")
    ]
    areas_by_tower[tower] = area_assets["Name"].tolist()

# Discover signal names: match StoredSignal, CalculatedSignal, or Signal; Path = parent (area) path
is_signal = items_df["Type"].str.contains("Signal", na=False)
signal_names = []
for tower in tower_names:
    for area in areas_by_tower[tower]:
        area_path = f"{EXAMPLE_ROOT} >> {tower} >> {area}"
        sigs = items_df[is_signal & (items_df["Path"] == area_path)]
        if not sigs.empty:
            signal_names = sigs["Name"].unique().tolist()
            break
    if signal_names:
        break

# Fallback: if the pulled tree has no signal rows (e.g. asset-only), search under the first area
if not signal_names and tower_names and areas_by_tower.get(tower_names[0]):
    first_area_path = f"{EXAMPLE_ROOT} >> {tower_names[0]} >> {areas_by_tower[tower_names[0]][0]}"
    search_results = spy.search(
        {"Path": first_area_path},
        recursive=False,
        all_properties=False,
    )
    signal_rows = search_results[search_results["Type"].str.contains("Signal", na=False)]
    if not signal_rows.empty:
        signal_names = signal_rows["Name"].unique().tolist()

if not signal_names:
    raise ValueError(
        "No signals found under any area in the Example tree. "
        "Ensure the source tree has at least one area with signals (StoredSignal or CalculatedSignal)."
    )

# Flatten to (area, tower) for iteration
all_areas = [(area, tower) for tower in tower_names for area in areas_by_tower[tower]]

print(f"Discovered {len(tower_names)} tower(s): {tower_names}")
print(f"Areas by tower: {areas_by_tower}")
print(f"Signal names ({len(signal_names)}): {signal_names}")
print(f"Total areas: {len(all_areas)}")

## 2. Build and push Data Share via metadata table

Use a **metadata DataFrame** and `spy.push(metadata=...)`. For assets, Path/Asset point to the parent; for conditions and signals, use Path = full path to the area and omit Asset so items attach under the existing area (no duplicate leaf assets). Conditions are **calculated** (Formula `condition(capsule(...), ...)`).

In [ ]:
def make_random_48h_capsules(
    num_capsules: int = 2,
    lookback_days: int = 14,
    duration_hours: int = 48,
    seed: Optional[int] = None,
) -> pd.DataFrame:
    """Build a DataFrame of non-overlapping 48h capsules in the last lookback_days."""
    if seed is not None:
        random.seed(seed)
    tz = timezone.utc
    end = datetime.now(tz)
    start = end - timedelta(days=lookback_days)
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    cap = pd.Timedelta(hours=duration_hours)
    # Latest possible start for a 48h capsule
    max_start = end_ts - cap
    if max_start <= start_ts:
        raise ValueError("Window too short for a 48h capsule")
    # Pick random start times; sort and enforce non-overlap
    starts = []
    for _ in range(num_capsules):
        s = start_ts + pd.Timedelta(seconds=random.uniform(0, (max_start - start_ts).total_seconds()))
        starts.append(s)
    starts.sort()
    # Ensure no overlap: shift each capsule so it starts after the previous one ends
    for i in range(1, num_capsules):
        prev_end = starts[i - 1] + cap
        if starts[i] < prev_end:
            starts[i] = prev_end
        if starts[i] + cap > end_ts:
            starts[i] = end_ts - cap
    rows = [{"Capsule Start": s, "Capsule End": s + cap} for s in starts]
    return pd.DataFrame(rows)

In [ ]:
CONDITION_NAME = "Shared Timerange"

# Build metadata rows: assets (root, towers, areas) then calculated conditions.
# Path = path where the item's parent asset resides; Asset = parent asset name (no duplication).
workbook_path = f"{WORKBOOK} >> Data Share Analysis"
rows = []

# 1. Root asset
rows.append({"Name": DATA_SHARE_ROOT, "Type": "Asset", "Path": "", "Asset": ""})

# 2. Tower assets (parent = root)
for tower in tower_names:
    rows.append({
        "Name": tower,
        "Type": "Asset",
        "Path": DATA_SHARE_ROOT,
        "Asset": DATA_SHARE_ROOT,
    })

# 3. Area assets (parent = tower). Path = parent path; Asset = parent name; path to area = Path >> Name only (no Asset in path).
for tower in tower_names:
    parent_path = f"{DATA_SHARE_ROOT} >> {tower}"
    for area in areas_by_tower[tower]:
        rows.append({
            "Name": area,
            "Type": "Asset",
            "Path": parent_path,
            "Asset": tower,
        })

# 4. Calculated conditions: Path = full path to area asset so item is placed at Path >> Name (under existing area, no duplicate).
for area, tower in all_areas:
    capsule_df = make_random_48h_capsules(
        num_capsules=NUM_CAPSULES_PER_AREA,
        lookback_days=LOOKBACK_DAYS,
        duration_hours=CAPSULE_DURATION_HOURS,
        seed=hash(area) % (2**32),
    )
    caps = ", ".join(
        f"capsule('{row['Capsule Start'].isoformat()}', '{row['Capsule End'].isoformat()}')"
        for _, row in capsule_df.iterrows()
    )
    formula = f"condition({caps})"
    area_path = f"{DATA_SHARE_ROOT} >> {tower} >> {area}"
    # Omit Asset so Seeq places at Path >> Name only (Path already points to area asset; Asset would add duplicate).
    rows.append({
        "Name": CONDITION_NAME,
        "Type": "Condition",
        "Formula": formula,
        "Path": area_path,
    })

metadata_df = pd.DataFrame(rows)
push_result = spy.push(
    metadata=metadata_df,
    workbook=workbook_path,
    datasource=DATASOURCE,
)

# Map (tower, area) -> condition ID (Path in result = full area path "Data Share >> Tower >> Area")
condition_ids = {}
id_col = "ID" if "ID" in push_result.columns else "Id"
condition_mask = push_result["Type"].str.contains("Condition", na=False)
condition_result = push_result.loc[condition_mask]
for _, row in condition_result.iterrows():
    path = row.get("Path", "")
    if path and id_col in row and pd.notna(row[id_col]):
        parts = [p.strip() for p in path.split(">>")]
        if len(parts) >= 3:
            tower, area = parts[-2], parts[-1]
            condition_ids[(tower, area)] = row[id_col]

print(f"Pushed {len(rows)} metadata rows (assets + conditions). Created {len(condition_ids)} Shared Timerange conditions.")
display(push_result)

## 4. Push calculated signals ($signal.within($condition)) via metadata

Build a metadata DataFrame of calculated signals (Formula + Formula Parameters by ID) and push. Path = full path to each area; Asset is omitted so signals attach under the existing area.

In [ ]:
# (tower, area, signal_name) -> source signal ID. Use ALL signals under each area (not just signal_names) so every tower gets signals.
id_col = "ID" if "ID" in items_df.columns else "Id"
is_signal = items_df["Type"].str.contains("Signal", na=False)
source_signal_ids = {}
for tower in tower_names:
    for area in areas_by_tower[tower]:
        area_path = f"{EXAMPLE_ROOT} >> {tower} >> {area}"
        sigs = items_df[is_signal & (items_df["Path"] == area_path)]
        for _, row in sigs.iterrows():
            signal_name = row["Name"]
            if id_col in row and pd.notna(row[id_col]):
                source_signal_ids[(tower, area, signal_name)] = row[id_col]

# Fallback: if items_df has no signal IDs, get IDs via spy.search per area
if not source_signal_ids and tower_names:
    for tower in tower_names:
        for area in areas_by_tower[tower]:
            area_path = f"{EXAMPLE_ROOT} >> {tower} >> {area}"
            found = spy.search({"Path": area_path}, recursive=False, all_properties=False)
            sigs = found[found["Type"].str.contains("Signal", na=False)]
            for _, r in sigs.iterrows():
                sid = r.get("ID", r.get("Id"))
                if pd.notna(sid):
                    source_signal_ids[(tower, area, r["Name"])] = sid

signal_rows = []
for (tower, area, signal_name), source_id in source_signal_ids.items():
    condition_id = condition_ids.get((tower, area))
    if not condition_id:
        continue
    # Path = full path to area; omit Asset so item is placed at Path >> Name (no duplicate area).
    area_path = f"{DATA_SHARE_ROOT} >> {tower} >> {area}"
    signal_rows.append({
        "Name": signal_name,
        "Type": "Signal",
        "Formula": "$signal.within($condition)",
        "Formula Parameters": {"$signal": source_id, "$condition": condition_id},
        "Path": area_path,
    })

if signal_rows:
    signals_metadata = pd.DataFrame(signal_rows)
    signals_push_result = spy.push(
        metadata=signals_metadata,
        workbook=workbook_path,
        datasource=DATASOURCE,
    )
    print(f"Pushed {len(signal_rows)} calculated signals.")
    display(signals_push_result)
else:
    print(f"No signal rows to push. condition_ids: {len(condition_ids)}, source_signal_ids: {len(source_signal_ids)}.")
    print("Re-run the discovery cell, then Section 2 (assets+conditions), then this cell. If condition_ids is 0, Section 2 push may have failed or returned a different structure.")

## 5. Apply ACL for the Data Share application (user)

Grant the configured application/user Read (and optionally Write/Manage) access to all assets and signals in the Data Share tree. Run this cell after the tree and items exist.

In [ ]:
# Find all items in the Data Share tree (recursive: at and below root)
data_share_items = spy.search(
    {"Path": DATA_SHARE_ROOT},
    workbook=f"{WORKBOOK} >> Data Share Analysis",
    recursive=True,
    all_properties=False,
)

if data_share_items.empty:
    print("No Data Share items found. Ensure the tree was pushed and workbook path is correct.")
else:
    acl_entries = [
        {"Username": DATA_SHARE_ACL_USER, "Read": True, "Write": True, "Manage": False}
    ]
    acl_result = spy.acl.push(data_share_items, acl_entries)
    print(f"Applied ACL to {len(data_share_items)} items for user/application: {DATA_SHARE_ACL_USER}")
    display(acl_result)

---
## Summary

- **Data Share** replicates the **entire asset tree** (root → top-level assets → areas → signals) using a metadata table and `spy.push(metadata=...)`. Structure is discovered from the source tree; no asset names are hardcoded.
- For conditions and signals, **Path** = full path to the area; **Asset** is omitted so items attach under the existing area (no duplicate leaf).
- Each area has a **Shared Timerange** calculated condition and calculated signals `$signal.within($condition)`; all signals under each area are included. *The random 48-hour capsules in this notebook are for testing only; in production, drive the time ranges from AF or from an ERP (e.g. only share windows the business has approved for sharing).*
- **ACL:** Create an application service principal in Seeq (Admin → Security → Service Accounts) first, then run the ACL cell to grant that identity access to the Data Share items.

See **docs/DATA_SHARE_README.md** for end-user documentation (what Data Share is, how to use it, and automation notes).